In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv("cleaned_games_2018_2022.csv")

### 1️⃣ Exploratory Data Analysis (EDA) ###
print("Dataset Shape:", df.shape)
print(df.info())

# Check missing values
print("\nMissing Values:\n", df.isnull().sum())

# Summary statistics
print("\nSummary Statistics:\n", df.describe())

# Distribution of Scores
plt.figure(figsize=(10,5))
sns.histplot(df['hometeamscoreft'], bins=20, kde=True, label='Home Team Score')
sns.histplot(df['awayteamscoreft'], bins=20, kde=True, color='red', label='Away Team Score')
plt.legend()
plt.title("Score Distribution")
plt.show()

### 2️⃣ Feature Engineering - Extract Team Statistics ###
# Convert scores to total points (Goals × 6 + Points)
df['hometeam_total_points'] = df['hometeamscoreft'].apply(lambda x: int(str(x).split(".")[0]) * 6 + int(str(x).split(".")[1]))
df['awayteam_total_points'] = df['awayteamscoreft'].apply(lambda x: int(str(x).split(".")[0]) * 6 + int(str(x).split(".")[1]))

# Win/Loss feature
df['home_win'] = (df['hometeam_total_points'] > df['awayteam_total_points']).astype(int)

# Team performance over time
team_stats = df.groupby('hometeam').agg({
    'hometeam_total_points': ['mean', 'std', 'max'],
    'home_win': ['mean']
}).reset_index()
team_stats.columns = ['hometeam', 'avg_points', 'std_points', 'max_points', 'win_rate']
print("\nTeam Statistics:\n", team_stats.head())


In [ ]:
df = df.merge(team_stats, on='hometeam', how='left')

# Define features and target variable
features = ['avg_points', 'std_points', 'max_points', 'win_rate', 'maxtemp', 'mintemp', 'rainfall']
X = df[features]
y = df['home_win']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Random forest

In [ ]:
# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predictions & Evaluation
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))




In [ ]:
class AFLPredictor:
    def __init__(self, transformer, estimator, team_stats_df):
        self.transformer = transformer
        self.estimator = estimator
        self.team_stats = team_stats_df

    def getTeamStats(self, homeTeam, awayTeam):
        # Define the features we're using
        features = ['avg_points', 'std_points', 'max_points', 'win_rate', 'maxtemp', 'mintemp', 'rainfall']

        # Get stats for home team
        home_team_stats = self.team_stats[self.team_stats['hometeam'] == homeTeam]

        # Handle case where team isn't in the data
        if home_team_stats.empty:
            raise ValueError(f"Home team {homeTeam} not found in data")

        # Get the latest weather data for prediction (assuming it's the most recent entry)
        latest_weather = self.team_stats[self.team_stats['hometeam'] == homeTeam][['maxtemp', 'mintemp', 'rainfall']].iloc[0]

        # Create a dataframe with the same feature names as used in training
        match_features = pd.DataFrame({
            'avg_points': [home_team_stats['avg_points'].values[0]],
            'std_points': [home_team_stats['std_points'].values[0]],
            'max_points': [home_team_stats['max_points'].values[0]],
            'win_rate': [home_team_stats['win_rate'].values[0]],
            'maxtemp': [latest_weather['maxtemp']],
            'mintemp': [latest_weather['mintemp']],
            'rainfall': [latest_weather['rainfall']]
        })

        return match_features

    def predict(self, homeTeam, awayTeam):
        stats = self.getTeamStats(homeTeam, awayTeam)

        # Scale the features
        stats_scaled = self.transformer.transform(stats)

        # Make the prediction
        prediction = self.estimator.predict(stats_scaled)[0]

        # Return the prediction (1 for home team win, 0 for away team win)
        return "Home Team (Win)" if prediction == 1 else "Away Team (Win)"

In [ ]:
# Example Prediction
home_team = "Carlton"
away_team = "West Coast"
afl_predictor = AFLPredictor(scaler, model, df)
print(f"Predicted winner: {afl_predictor.predict(home_team, away_team)}")


In [ ]:
# Example Prediction
home_team = "Greater Western Sydney"
away_team = "Western Bulldogs"
afl_predictor = AFLPredictor(scaler, model, df)
print(f"Predicted winner: {afl_predictor.predict(home_team, away_team)}")


Linear SVC

In [ ]:
from sklearn.svm import LinearSVC

svm_clf = LinearSVC(C=0.001, loss='hinge')

In [ ]:
svm_clf.fit(X_train, y_train)

In [ ]:
y_pred = svm_clf.predict(X_test)
print(y_pred)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# Example Prediction
home_team = "Greater Western Sydney"
away_team = "Western Bulldogs"
afl_predictor = AFLPredictor(scaler, svm_clf, df)
print(f"Predicted winner: {afl_predictor.predict(home_team, away_team)}")


In [ ]:
# Example Prediction
home_team = "Carlton"
away_team = "West Coast"
afl_predictor = AFLPredictor(scaler, svm_clf, df)
print(f"Predicted winner: {afl_predictor.predict(home_team, away_team)}")


## XGBRegressor

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Load dataset
df = pd.read_csv("/content/cleaned_games_2018_2022.csv")

def compute_final_score(score):
    goals, points = map(int, str(score).split('.'))
    return (goals * 6) + points

df['home_score'] = df['hometeamscoreft'].apply(compute_final_score)
df['away_score'] = df['awayteamscoreft'].apply(compute_final_score)

df.head()
# Encode categorical variables (home team, away team, venue)
label_encoders = {}
categorical_columns = ['hometeam', 'awayteam', 'venue','round']

for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le  # Save encoders for later use

# Select features and target
features = ['year', 'round', 'hometeam', 'awayteam', 'venue', 'maxtemp', 'mintemp']
target = ['home_score', 'away_score']


X = df[features]
y = df[target]

X.dtypes

# Split data into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost Model
model = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Convert predictions to integers (rounding)
y_pred_int = np.round(y_pred)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred_int)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_int))
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.4f}")  # Closer to 1 means better model fit


In [ ]:
def predict_match(home_team, away_team, round, venue, year=2024, maxtemp=20, mintemp=10):
    # Encode categorical values using saved LabelEncoders
    home_team_encoded = label_encoders['hometeam'].transform([home_team])[0]
    away_team_encoded = label_encoders['awayteam'].transform([away_team])[0]
    venue_encoded = label_encoders['venue'].transform([venue])[0]
    round_encoded = label_encoders['round'].transform([round])[0]
    # Create input array
    match_features = np.array([[year, round_encoded,home_team_encoded, away_team_encoded, venue_encoded, maxtemp, mintemp]])

    # Predict scores
    predicted_scores = model.predict(match_features)

    # Round to nearest integer
    home_score = round(predicted_scores[0][0])
    away_score = round(predicted_scores[0][1])

    print(f"Predicted Score for {home_team} vs {away_team}:")
    print(f"{home_team}: {home_score}")
    print(f"{away_team}: {away_score}")

# Example Prediction: Geelong Cats vs Richmond at MCG
predict_match("Richmond", "Carlton","R1", "M.C.G.", year=2024, maxtemp=	28.7, mintemp=14)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# Load dataset
df = pd.read_csv("/content/cleaned_games_2018_2022.csv")

# Function to compute final AFL score
def compute_final_score(score):
    goals, points = map(int, str(score).split('.'))
    return (goals * 6) + points

df['home_score'] = df['hometeamscoreft'].apply(compute_final_score)
df['away_score'] = df['awayteamscoreft'].apply(compute_final_score)

# Encode categorical variables
label_encoders = {}
categorical_columns = ['hometeam', 'awayteam', 'venue', 'round']

for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le  # Save encoders for later use

# Select features and target
features = ['year', 'round', 'hometeam', 'awayteam', 'venue', 'maxtemp', 'mintemp']
target = ['home_score', 'away_score']

X = df[features]
y = df[target]

# Split data into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost Model
model = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Convert predictions to integers (rounding)
y_pred_int = np.round(y_pred)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred_int)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_int))
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

# Prediction function
def predict_match(home_team, away_team, match_round, venue, year=2024, maxtemp=20, mintemp=10):
    # Encode categorical values using saved LabelEncoders
    home_team_encoded = label_encoders['hometeam'].transform([home_team])[0]
    away_team_encoded = label_encoders['awayteam'].transform([away_team])[0]
    venue_encoded = label_encoders['venue'].transform([venue])[0]
    round_encoded = label_encoders['round'].transform([match_round])[0]

    # Create input array
    match_features = np.array([[year, round_encoded, home_team_encoded, away_team_encoded, venue_encoded, maxtemp, mintemp]])

    # Predict scores
    predicted_scores = model.predict(match_features)

    # Round to nearest integer
    home_score = int(round(predicted_scores[0][0]))
    away_score = int(round(predicted_scores[0][1]))

    print(f"Predicted Score for {home_team} vs {away_team}:")
    print(f"{home_team}: {home_score}")
    print(f"{away_team}: {away_score}")

# Example Prediction: Richmond vs Carlton at MCG
predict_match("Richmond", "Carlton", "R1", "M.C.G.", year=2024, maxtemp=28.7, mintemp=14)


## Hybrid

In [ ]:
import math
from decimal import Decimal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import math

# Glicko-2 implementation
class Glicko2:
    def __init__(self, rating=1500, rd=350, vol=0.06):
        """
        Initialize a player with the given rating, rating deviation, and volatility.
        """
        self.rating = rating
        self.rd = rd  # Rating Deviation
        self.vol = vol  # Volatility
        self._tau = 0.5  # System constant, smaller = slower volatility changes
        self._scale_factor = 173.7178  # Used to convert ratings to Glicko-2 scale

    def _g(self, rd):
        """
        This function calculates g(RD) as defined by Glickman.
        """
        return 1.0 / math.sqrt(1.0 + 3.0 * rd * rd / (math.pi * math.pi))

    def _E(self, rating, other_rating, other_rd):
        """
        Expected score of a player against an opponent.
        """
        return 1.0 / (1 + math.exp(-self._g(other_rd) * (rating - other_rating) / 400.0))

    def expected_score(self, opponent):
        """
        Calculate expected score against an opponent.
        """
        return self._E(self.rating, opponent.rating, opponent.rd)

    def update(self, opponent, outcome):
        """
        Update player's rating after a match against opponent with outcome.
        Outcome should be 1.0 for win, 0.5 for draw, 0.0 for loss.
        """
        # Convert ratings to Glicko-2 scale
        r = (self.rating - 1500) / self._scale_factor
        rd = self.rd / self._scale_factor

        opponent_r = (opponent.rating - 1500) / self._scale_factor
        opponent_rd = opponent.rd / self._scale_factor

        # Compute v
        g_opponent = self._g(opponent_rd)
        E = self._E(r, opponent_r, opponent_rd)
        v = 1.0 / (g_opponent * g_opponent * E * (1 - E))

        # Compute delta
        delta = v * g_opponent * (outcome - E)

        # Compute new volatility
        a = math.log(self.vol * self.vol)
        A = a
        B = 0.0

        if delta * delta > rd * rd + v:
            B = math.log(delta * delta - rd * rd - v)
        else:
            k = 1
            while self._f(a - k * self._tau, delta, rd, v) < 0:
                k *= 2
            B = a - k * self._tau

        f_A = self._f(A, delta, rd, v)
        f_B = self._f(B, delta, rd, v)

        while abs(B - A) > 0.000001:
            C = A + (A - B) * f_A / (f_B - f_A)
            f_C = self._f(C, delta, rd, v)

            if f_C * f_B <= 0:
                A = B
                f_A = f_B
            else:
                f_A /= 2.0

            B = C
            f_B = f_C

        new_vol = math.exp(A / 2.0)

        # Update rating deviation
        rd_star = math.sqrt(rd * rd + new_vol * new_vol)
        new_rd = 1.0 / math.sqrt(1.0 / (rd_star * rd_star) + 1.0 / v)

        # Update rating
        new_r = r + new_rd * new_rd * g_opponent * (outcome - E)

        # Convert back to original scale
        self.rating = new_r * self._scale_factor + 1500
        self.rd = new_rd * self._scale_factor
        self.vol = new_vol



    def _f(self, x, delta, rd, v):
        """
        Internal function used for volatility calculation.
        """
        x = Decimal(x)
        delta = Decimal(delta)
        rd = Decimal(rd)
        v = Decimal(v)
        ex = Decimal(math.exp(x))
        num1 = ex * (delta * delta - rd * rd - v - ex)
        denom1 = Decimal(2.0) * (rd * rd + v + ex) * (rd * rd + v + ex)  # Convert 2.0 to Decimal
        return (x - Decimal(math.log(self.vol * self.vol))) / (Decimal(self._tau * self._tau)) - num1 / denom1
    def get_rating(self):
        """
        Return the current rating.
        """
        return self.rating

    def get_rating_deviation(self):
        """
        Return the current rating deviation.
        """
        return self.rd

    def get_volatility(self):
        """
        Return the current volatility.
        """
        return self.vol

# Main processing and model training
def build_hybrid_model(df):
    """
    Build a hybrid model using Glicko-2 ratings as features for Random Forest

    Parameters:
    df (pandas.DataFrame): DataFrame with match results

    Returns:
    tuple: (trained_win_model, trained_score_model, team_ratings, scaler)
    """
    print("Building hybrid prediction model...")

    # Initialize Glicko ratings for teams
    team_ratings = {}
    for team in set(list(df['hometeam'].unique()) + list(df['awayteam'].unique())):
        team_ratings[team] = Glicko2()

    # Process matches chronologically to update team ratings
    df_sorted = df.sort_values('date')

    # Function to convert AFL score format to final score
    def compute_final_score(score):
        try:
            goals, points = map(int, str(score).split('.'))
            return (goals * 6) + points
        except:
            # Handle potential formatting issues
            return 0

    # Add columns for computed scores
    df_sorted['home_score'] = df_sorted['hometeamscoreft'].apply(compute_final_score)
    df_sorted['away_score'] = df_sorted['awayteamscoreft'].apply(compute_final_score)
    df_sorted['score_diff'] = df_sorted['home_score'] - df_sorted['away_score']
    df_sorted['home_win'] = (df_sorted['score_diff'] > 0).astype(int)

    # Create feature dataframe for training
    features_df = pd.DataFrame()

    # Process matches and build features
    for idx, row in df_sorted.iterrows():
        home_team = row['hometeam']
        away_team = row['awayteam']

        # For the first iteration, just record the match info without prediction
        if home_team not in team_ratings:
            team_ratings[home_team] = Glicko2()
        if away_team not in team_ratings:
            team_ratings[away_team] = Glicko2()

        # Get pre-match ratings
        home_rating = team_ratings[home_team].get_rating()
        home_rd = team_ratings[home_team].get_rating_deviation()
        away_rating = team_ratings[away_team].get_rating()
        away_rd = team_ratings[away_team].get_rating_deviation()

        # Store features for this match
        match_features = {
            'match_id': idx,
            'home_team': home_team,
            'away_team': away_team,
            'home_rating': home_rating,
            'home_rd': home_rd,
            'away_rating': away_rating,
            'away_rd': away_rd,
            'rating_diff': home_rating - away_rating,
            'rd_diff': home_rd - away_rd,
            'expected_home_win': team_ratings[home_team].expected_score(team_ratings[away_team]),
        }

        # Add weather features if available
        for feature in ['maxtemp', 'mintemp', 'rainfall']:
            if feature in row:
                match_features[feature] = row[feature]

        # Append to features dataframe
        features_df = pd.concat([features_df, pd.DataFrame([match_features])], ignore_index=True)

        # Determine match result
        if row['home_score'] > row['away_score']:
            home_result, away_result = 1, 0  # Home wins
        elif row['home_score'] < row['away_score']:
            home_result, away_result = 0, 1  # Away wins
        else:
            home_result, away_result = 0.5, 0.5  # Draw

        # Update Glicko-2 ratings based on results
        home_team_glicko = team_ratings[home_team]
        away_team_glicko = team_ratings[away_team]
        home_team_glicko.update(away_team_glicko, home_result)
        away_team_glicko.update(home_team_glicko, away_result)

    # Merge with actual outcomes
    model_df = pd.merge(features_df,
                       df_sorted[['hometeam', 'awayteam', 'home_score', 'away_score', 'score_diff', 'home_win']],
                       left_on=['home_team', 'away_team'],
                       right_on=['hometeam', 'awayteam'])

    # Define features for the model
    feature_columns = ['home_rating', 'home_rd', 'away_rating', 'away_rd', 'rating_diff', 'rd_diff', 'expected_home_win']

    # Add weather features if available
    for feature in ['maxtemp', 'mintemp', 'rainfall']:
        if feature in model_df.columns:
            feature_columns.append(feature)

    # Prepare features and targets
    X = model_df[feature_columns]
    y_win = model_df['home_win']
    y_margin = model_df['score_diff']
    y_home_score = model_df['home_score']
    y_away_score = model_df['away_score']

    # Preprocess features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Split data for win prediction
    X_train_win, X_test_win, y_train_win, y_test_win = train_test_split(
        X_scaled, y_win, test_size=0.2, random_state=42)

    # Train win prediction model
    print("Training win prediction model...")
    win_model = RandomForestClassifier(n_estimators=100, random_state=42)
    win_model.fit(X_train_win, y_train_win)

    # Evaluate win prediction
    win_pred = win_model.predict(X_test_win)
    win_accuracy = accuracy_score(y_test_win, win_pred)
    print(f"Win prediction accuracy: {win_accuracy:.4f}")
    print(classification_report(y_test_win, win_pred))

    # Split data for score prediction
    X_train_score, X_test_score, y_train_score, y_test_score = train_test_split(
        X_scaled, model_df[['home_score', 'away_score']], test_size=0.2, random_state=42)

    # Train score prediction model
    print("Training score prediction model...")
    score_model = RandomForestRegressor(n_estimators=100, random_state=42)
    score_model.fit(X_train_score, y_train_score)

    # Evaluate score prediction
    score_pred = score_model.predict(X_test_score)
    home_score_rmse = mean_squared_error(y_test_score['home_score'], score_pred[:, 0], squared=False)
    away_score_rmse = mean_squared_error(y_test_score['away_score'], score_pred[:, 1], squared=False)
    print(f"Home score RMSE: {home_score_rmse:.2f} points")
    print(f"Away score RMSE: {away_score_rmse:.2f} points")

    # Feature importance for win model
    feature_importances_win = pd.DataFrame(
        win_model.feature_importances_,
        index=feature_columns,
        columns=['importance']
    ).sort_values('importance', ascending=False)

    print("\nFeature importances for win prediction:")
    print(feature_importances_win)

    return win_model, score_model, team_ratings, scaler, feature_columns

# Prediction class
class AFLHybridPredictor:
    def __init__(self, win_model, score_model, team_ratings, scaler, feature_columns):
        """
        Initialize the hybrid predictor with trained models, team ratings and preprocessors
        """
        self.win_model = win_model
        self.score_model = score_model
        self.team_ratings = team_ratings
        self.scaler = scaler
        self.feature_columns = feature_columns

    def predict_match(self, home_team, away_team, weather_data=None):
        """
        Predict outcome of a match between home_team and away_team

        Parameters:
        home_team (str): Name of home team
        away_team (str): Name of away team
        weather_data (dict, optional): Weather conditions for the match

        Returns:
        dict: Prediction results
        """
        # Check if teams exist in ratings
        if home_team not in self.team_ratings or away_team not in self.team_ratings:
            print(f"Error: Team(s) not found in ratings database")
            return None

        # Get team ratings
        home_rating = self.team_ratings[home_team].get_rating()
        home_rd = self.team_ratings[home_team].get_rating_deviation()
        away_rating = self.team_ratings[away_team].get_rating()
        away_rd = self.team_ratings[away_team].get_rating_deviation()

        # Calculate expected outcome from Glicko
        expected_home_win = self.team_ratings[home_team].expected_score(self.team_ratings[away_team])

        # Create feature array
        features = {
            'home_rating': home_rating,
            'home_rd': home_rd,
            'away_rating': away_rating,
            'away_rd': away_rd,
            'rating_diff': home_rating - away_rating,
            'rd_diff': home_rd - away_rd,
            'expected_home_win': expected_home_win
        }

        # Add weather data if provided
        if weather_data:
            for key, value in weather_data.items():
                if key in self.feature_columns:
                    features[key] = value

        # Create dataframe and ensure all needed columns are present
        X = pd.DataFrame([features])
        for col in self.feature_columns:
            if col not in X:
                X[col] = 0

        # Keep only necessary columns in the right order
        X = X[self.feature_columns]

        # Scale features
        X_scaled = self.scaler.transform(X)

        # Make predictions
        win_prob = self.win_model

In [ ]:
# Example usage of the AFL Hybrid Prediction Model

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assume df contains your AFL match data with required columns
# If not, replace with your actual data loading code
df = pd.read_csv("cleaned_games_2018_2022.csv")

sample_df = pd.DataFrame(df)

# Build the hybrid model with our sample data
win_model, score_model, team_ratings, scaler, feature_columns = build_hybrid_model(df)

# Create the predictor
predictor = AFLHybridPredictor(win_model, score_model, team_ratings, scaler, feature_columns)

# Example 1: Basic prediction
print("\n--- Example 1: Basic Match Prediction ---")
home_team = "Richmond"
away_team = "Collingwood"
prediction = predictor.predict_match(home_team, away_team)

# Example 2: Prediction with weather data
print("\n--- Example 2: Prediction with Weather Data ---")
weather_data = {
    'maxtemp': 25,
    'mintemp': 15,
    'rainfall': 0  # No rain
}
prediction_with_weather = predictor.predict_match(home_team, away_team, weather_data)

# Example 3: Season simulation
print("\n--- Example 3: Season Simulation ---")
# Define upcoming matches
upcoming_matches = [
    {"home": "Richmond", "away": "Hawthorn", "weather": {"maxtemp": 22, "mintemp": 14, "rainfall": 0}},
    {"home": "Collingwood", "away": "West Coast", "weather": {"maxtemp": 26, "mintemp": 16, "rainfall": 1}},
    {"home": "Geelong", "away": "Sydney", "weather": {"maxtemp": 20, "mintemp": 12, "rainfall": 3}},
    {"home": "Melbourne", "away": "Carlton", "weather": {"maxtemp": 24, "mintemp": 15, "rainfall": 0}}
]

# Simulate each match
print("Simulating upcoming matches:")
for match in upcoming_matches:
    result = predictor.predict_match(match["home"], match["away"], match["weather"])
    print(f"\n{match['home']} vs {match['away']}:")
    print(f"Predicted winner: {result['predicted_winner']}")
    print(f"Predicted scores: {match['home']} {result['predicted_home_score']:.1f} - {result['predicted_away_score']:.1f} {match['away']}")
    print(f"Win probability: {result['win_probability']:.1%}")

# Example 4: Visualize team ratings
print("\n--- Example 4: Team Ratings Visualization ---")
team_names = list(team_ratings.keys())
team_rating_values = [team_ratings[team].get_rating() for team in team_names]
team_rd_values = [team_ratings[team].get_rating_deviation() for team in team_names]

# Create a dataframe for plotting
ratings_df = pd.DataFrame({
    'Team': team_names,
    'Rating': team_rating_values,
    'RD': team_rd_values
})
ratings_df = ratings_df.sort_values('Rating', ascending=False)

# Plot team ratings
plt.figure(figsize=(12, 8))
bar_plot = sns.barplot(x='Rating', y='Team', data=ratings_df)
plt.title('AFL Team Glicko-2 Ratings')
plt.xlabel('Rating')
plt.ylabel('Team')
plt.tight_layout()
plt.show()

# Example 5: Team matchup visualization
print("\n--- Example 5: Team Matchup Analysis ---")
teams_to_compare = ['Richmond', 'Collingwood', 'Geelong', 'Melbourne']

# Create matchup matrix
matchup_data = []
for home in teams_to_compare:
    for away in teams_to_compare:
        if home != away:
            result = predictor.predict_match(home, away)
            matchup_data.append({
                'Home': home,
                'Away': away,
                'WinProb': result['win_probability']
            })

matchup_df = pd.DataFrame(matchup_data)
matchup_matrix = matchup_df.pivot(index='Home', columns='Away', values='WinProb')

# Plot matchup heatmap
plt.figure(figsize=(10, 8))
heatmap = sns.heatmap(matchup_matrix, annot=True, cmap="YlGnBu", fmt='.2f',
                     vmin=0, vmax=1, cbar_kws={'label': 'Home Team Win Probability'})
plt.title('Team Matchup Analysis - Home Team Win Probability')
plt.tight_layout()
plt.show()

# Example 6: Feature importance visualization
print("\n--- Example 6: Feature Importance Analysis ---")
feature_imp = pd.DataFrame(win_model.feature_importances_,
                          index=feature_columns,
                          columns=['Importance']).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feature_imp.Importance, y=feature_imp.index)
plt.title('Feature Importance for Win Prediction')
plt.tight_layout()
plt.show()

print("\nModel is ready for predictions. Use predictor.predict_match(home_team, away_team, weather_data) for future matches.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("cleaned_games_2018_2022.csv")
sample_df = pd.DataFrame(df)

In [ ]:
win_model, score_model, team_ratings, scaler, feature_columns = build_hybrid_model(df)
predictor = AFLHybridPredictor(win_model, score_model, team_ratings, scaler, feature_columns)

CatBoost


In [ ]:
!pip install catboost


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 84.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2


In [ ]:
!pip install --upgrade --force-reinstall numpy==1.23.5 catboost


  Using cached catboost-1.2.7-cp311-cp311-manylinux2014_x86_64.whl.metadata (1.2 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 50.0 MB/s eta 0:00:00
Using cached catboost-1.2.7-cp311-cp311-manylinux2014_x86_64.whl (98.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.2/326.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("/content/cleaned_games_2018_2022.csv")

def compute_final_score(score):
    goals, points = map(int, str(score).split('.'))
    return (goals * 6) + points

df['home_score'] = df['hometeamscoreft'].apply(compute_final_score)
df['away_score'] = df['awayteamscoreft'].apply(compute_final_score)

# Assume you already label encoded
categorical_features = ['hometeam', 'awayteam', 'venue', 'round']
features = ['year', 'round', 'hometeam', 'awayteam', 'venue', 'maxtemp', 'mintemp']

X = df[features]
y_home = df['home_score']
y_away = df['away_score']

X_train, X_test, y_train_home, y_test_home = train_test_split(X, y_home, test_size=0.2, random_state=42)
_, _, y_train_away, y_test_away = train_test_split(X, y_away, test_size=0.2, random_state=42)

# Train two models
model_home = CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6, random_seed=42, verbose=0)
model_away = CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6, random_seed=42, verbose=0)

model_home.fit(X_train, y_train_home, cat_features=[features.index(col) for col in categorical_features])
model_away.fit(X_train, y_train_away, cat_features=[features.index(col) for col in categorical_features])

# Predictions
y_pred_home = model_home.predict(X_test)
y_pred_away = model_away.predict(X_test)

# Rounding
y_pred_home_int = np.round(y_pred_home)
y_pred_away_int = np.round(y_pred_away)

# Evaluation
mae_home = mean_absolute_error(y_test_home, y_pred_home_int)
mae_away = mean_absolute_error(y_test_away, y_pred_away_int)

rmse_home = np.sqrt(mean_squared_error(y_test_home, y_pred_home_int))
rmse_away = np.sqrt(mean_squared_error(y_test_away, y_pred_away_int))

r2_home = r2_score(y_test_home, y_pred_home)
r2_away = r2_score(y_test_away, y_pred_away)

print(f"🏠 Home Score - MAE: {mae_home:.2f}, RMSE: {rmse_home:.2f}, R²: {r2_home:.4f}")
print(f"🧳 Away Score - MAE: {mae_away:.2f}, RMSE: {rmse_away:.2f}, R²: {r2_away:.4f}")


🏠 Home Score - MAE: 20.36, RMSE: 24.94, R²: 0.0549
🧳 Away Score - MAE: 18.62, RMSE: 23.63, R²: 0.1471


In [ ]:
mae = mean_absolute_error(y_test, y_pred_int)
print(f"MAE: {mae}")
rmse = np.sqrt(mean_squared_error(y_test, y_pred_int))
print(f"RMSE: {rmse}")
r2 = r2_score(y_test, y_pred)
print(f"R² Score: {r2}")


Merge player statistics


In [ ]:
import pandas as pd

# Load player data
player_df = pd.read_csv("/content/stats.csv")

# Aggregate player statistics by GameId and Team
agg_player_stats = player_df.groupby(['GameId', 'Team']).agg({
    'Disposals': 'sum',
    'Kicks': 'sum',
    'Marks': 'sum',
    'Handballs': 'sum',
    'Goals': 'sum',
    'Behinds': 'sum',
    'HitOuts': 'sum',
    'Tackles': 'sum',
    'Inside50s': 'sum',
    'Clearances': 'sum',
    'Rebounds': 'sum',
    # Add more as needed!
}).reset_index()

print(agg_player_stats.head())


     GameId             Team  Disposals  Kicks  Marks  Handballs  Goals  \
0  2012EF01        Fremantle        325    211     94        114     14   
1  2012EF01          Geelong        321    182     52        139     11   
2  2012EF02  North Melbourne        289    170     57        119      9   
3  2012EF02       West Coast        358    231    112        127     24   
4  2012GF01         Hawthorn        336    194     56        142     11   

   Behinds  HitOuts  Tackles  Inside50s  Clearances  Rebounds  
0        8       44       77         47          34        35  
1       11       31       86         54          38        25  
2       10       29       36         43          42        41  
3       13       62       38         71          43        28  
4       13       60       84         61          58        26  


In [ ]:
# Assuming your match data is already loaded as df_matches
df_matches = pd.read_csv("/content/cleaned_games_2018_2022.csv")

# First, make sure GameId exists and matches between datasets
# You may need to create GameId from your match data (e.g., Year + Round + match number)

# Merge home team stats
df_matches = df_matches.merge(
    agg_player_stats,
    left_on=['gameid', 'hometeam'],
    right_on=['GameId', 'Team'],
    how='left',
    suffixes=('', '_home')
)

# Merge away team stats
df_matches = df_matches.merge(
    agg_player_stats,
    left_on=['GameId', 'awayteam'],
    right_on=['GameId', 'Team'],
    how='left',
    suffixes=('', '_away')
)

print(df_matches.head())


      gameid  year round        date       hometeam         awayteam  \
0  2018R0101  2018    R1  2018-03-22       Richmond          Carlton   
1  2018R0102  2018    R1  2018-03-23       Essendon         Adelaide   
2  2018R0103  2018    R1  2018-03-24       St Kilda   Brisbane Lions   
3  2018R0104  2018    R1  2018-03-24  Port Adelaide        Fremantle   
4  2018R0105  2018    R1  2018-03-24     Gold Coast  North Melbourne   

   hometeamscoreft  awayteamscoreft             venue  attendance  ...  \
0            17.19            15.50            M.C.G.     90151.0  ...   
1            14.15            12.15         Docklands     43016.0  ...   
2            16.11            12.10         Docklands     23731.0  ...   
3            16.14             9.60     Adelaide Oval     38324.0  ...   
4             7.13             5.90  Cazaly's Stadium      3722.0  ...   

   Kicks_away  Marks_away  Handballs_away Goals_away Behinds_away  \
0         207          88             169         15 

In [ ]:
features = [
    'year', 'round', 'hometeam', 'awayteam', 'venue', 'maxtemp', 'mintemp',
    # Home team player stats
    'Disposals', 'Kicks', 'Marks', 'Handballs', 'Goals', 'Behinds', 'HitOuts', 'Tackles', 'Inside50s', 'Clearances', 'Rebounds',
    # Away team player stats
    'Disposals_away', 'Kicks_away', 'Marks_away', 'Handballs_away', 'Goals_away', 'Behinds_away', 'HitOuts_away', 'Tackles_away', 'Inside50s_away', 'Clearances_away', 'Rebounds_away',
]


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor


def compute_final_score(score):
    goals, points = map(int, str(score).split('.'))
    return (goals * 6) + points

df_matches['home_score'] = df_matches['hometeamscoreft'].apply(compute_final_score)
df_matches['away_score'] = df_matches['awayteamscoreft'].apply(compute_final_score)

# Assume you already label encoded
categorical_features = ['hometeam', 'awayteam', 'venue', 'round']
X = df_matches[features]
y_home = df_matches['home_score']
y_away = df_matches['away_score']

X_train, X_test, y_train_home, y_test_home = train_test_split(X, y_home, test_size=0.2, random_state=42)
_, _, y_train_away, y_test_away = train_test_split(X, y_away, test_size=0.2, random_state=42)

# Train two models
model_home = CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6, random_seed=42, verbose=0)
model_away = CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6, random_seed=42, verbose=0)

model_home.fit(X_train, y_train_home, cat_features=[features.index(col) for col in categorical_features])
model_away.fit(X_train, y_train_away, cat_features=[features.index(col) for col in categorical_features])

# Predictions
y_pred_home = model_home.predict(X_test)
y_pred_away = model_away.predict(X_test)

# Rounding
y_pred_home_int = np.round(y_pred_home)
y_pred_away_int = np.round(y_pred_away)

# Evaluation
mae_home = mean_absolute_error(y_test_home, y_pred_home_int)
mae_away = mean_absolute_error(y_test_away, y_pred_away_int)

rmse_home = np.sqrt(mean_squared_error(y_test_home, y_pred_home_int))
rmse_away = np.sqrt(mean_squared_error(y_test_away, y_pred_away_int))

r2_home = r2_score(y_test_home, y_pred_home)
r2_away = r2_score(y_test_away, y_pred_away)

print(f"🏠 Home Score - MAE: {mae_home:.2f}, RMSE: {rmse_home:.2f}, R²: {r2_home:.4f}")
print(f"🧳 Away Score - MAE: {mae_away:.2f}, RMSE: {rmse_away:.2f}, R²: {r2_away:.4f}")

🏠 Home Score - MAE: 2.76, RMSE: 3.62, R²: 0.9803
🧳 Away Score - MAE: 2.74, RMSE: 3.80, R²: 0.9786


In [ ]:

predict_match("Richmond", "Port Adelaide", "R1", "Adelaide Oval", year=2018, maxtemp=28.7, mintemp=14)

Predicted Score:
Richmond: 81
Port Adelaide: 75


(81, 75)

In [ ]:
import pandas as pd

# Load player data
player_df = pd.read_csv("/content/stats.csv")

# Aggregate player statistics by GameId and Team
agg_player_stats = player_df.groupby(['GameId', 'Team']).agg({
    'Disposals': 'sum',
    'Kicks': 'sum',
    'Marks': 'sum',
    'Handballs': 'sum',
    'Goals': 'sum',
    'Behinds': 'sum',
    'HitOuts': 'sum',
    'Tackles': 'sum',
    'Inside50s': 'sum',
    'Clearances': 'sum',
    'Rebounds': 'sum',
    # Add more as needed!
}).reset_index()

print(agg_player_stats.head())

     GameId             Team  Disposals  Kicks  Marks  Handballs  Goals  \
0  2012EF01        Fremantle        325    211     94        114     14   
1  2012EF01          Geelong        321    182     52        139     11   
2  2012EF02  North Melbourne        289    170     57        119      9   
3  2012EF02       West Coast        358    231    112        127     24   
4  2012GF01         Hawthorn        336    194     56        142     11   

   Behinds  HitOuts  Tackles  Inside50s  Clearances  Rebounds  
0        8       44       77         47          34        35  
1       11       31       86         54          38        25  
2       10       29       36         43          42        41  
3       13       62       38         71          43        28  
4       13       60       84         61          58        26  


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder

# Assuming your match data is already loaded as df_matches
df_matches = pd.read_csv("/content/cleaned_games_2018_2022.csv")

# First, make sure GameId exists and matches between datasets
# You may need to create GameId from your match data (e.g., Year + Round + match number)

# Merge home team stats
df_matches = df_matches.merge(
    agg_player_stats,
    left_on=['gameid', 'hometeam'],
    right_on=['GameId', 'Team'],
    how='left',
    suffixes=('', '_home')
)

# Merge away team stats
df_matches = df_matches.merge(
    agg_player_stats,
    left_on=['GameId', 'awayteam'],
    right_on=['GameId', 'Team'],
    how='left',
    suffixes=('', '_away')
)

print(df_matches.head())

# Ensure columns are strings
categorical_columns = ['hometeam', 'awayteam', 'venue', 'round']
for col in categorical_columns:
    df_matches[col] = df_matches[col].astype(str)

# ✅ Step 1: Encode categorical variables (before training!)
label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    le.fit(df_matches[col])
    df_matches[col] = le.transform(df_matches[col])
    label_encoders[col] = le

# ✅ Step 2: Compute final scores
def compute_final_score(score):
    goals, points = map(int, str(score).split('.'))
    return (goals * 6) + points

df_matches['home_score'] = df_matches['hometeamscoreft'].apply(compute_final_score)
df_matches['away_score'] = df_matches['awayteamscoreft'].apply(compute_final_score)

# ✅ Step 3: Define features
metadata_features = ['year', 'round', 'hometeam', 'awayteam', 'venue', 'maxtemp', 'mintemp']
player_stats = ['Disposals', 'Kicks', 'Marks', 'Handballs', 'Goals', 'Behinds', 'HitOuts',
                'Tackles', 'Inside50s', 'Clearances', 'Rebounds', 'Disposals_away', 'Kicks_away',
                'Marks_away', 'Handballs_away', 'Goals_away', 'Behinds_away', 'HitOuts_away',
                'Tackles_away', 'Inside50s_away', 'Clearances_away', 'Rebounds_away']

X_metadata = df_matches[metadata_features]

# ✅ Step 4: Train models for player stats
stat_models = {}
for stat in player_stats:
    y_stat = df_matches[stat]
    model = CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=42, verbose=0)
    model.fit(X_metadata, y_stat, cat_features=[metadata_features.index(col) for col in categorical_columns])
    stat_models[stat] = model

print("✅ Player stat models trained.")

# ✅ Step 5: Predict team stats for a new match
def predict_team_stats(home_team_name, away_team_name, round_name, venue_name, year=2024, maxtemp=20, mintemp=10):
    home_team_encoded = label_encoders['hometeam'].transform([home_team_name])[0]
    away_team_encoded = label_encoders['awayteam'].transform([away_team_name])[0]
    round_encoded = label_encoders['round'].transform([round_name])[0]
    venue_encoded = label_encoders['venue'].transform([venue_name])[0]

    input_metadata = pd.DataFrame({
        'year': [year],
        'round': [round_encoded],
        'hometeam': [home_team_encoded],
        'awayteam': [away_team_encoded],
        'venue': [venue_encoded],
        'maxtemp': [maxtemp],
        'mintemp': [mintemp],
    })

    predicted_stats = {}
    for stat, model in stat_models.items():
        predicted_value = model.predict(input_metadata)[0]
        predicted_stats[stat] = predicted_value

    return predicted_stats

# ✅ Step 6: Train final score models
X_full = pd.concat([X_metadata, df_matches[player_stats]], axis=1)
y_home = df_matches['home_score']
y_away = df_matches['away_score']

X_train, X_test, y_train_home, y_test_home = train_test_split(X_full, y_home, test_size=0.2, random_state=42)
_, _, y_train_away, y_test_away = train_test_split(X_full, y_away, test_size=0.2, random_state=42)

model_home = CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6, random_seed=42, verbose=0)
model_away = CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6, random_seed=42, verbose=0)

model_home.fit(X_train, y_train_home, cat_features=[metadata_features.index(col) for col in categorical_columns])
model_away.fit(X_train, y_train_away, cat_features=[metadata_features.index(col) for col in categorical_columns])

print("✅ Final score models trained.")

# ✅ Step 7: Final match prediction function
def predict_match(home_team_name, away_team_name, round_name, venue_name, year=2024, maxtemp=20, mintemp=10):
    predicted_stats = predict_team_stats(home_team_name, away_team_name, round_name, venue_name, year, maxtemp, mintemp)

    home_team_encoded = label_encoders['hometeam'].transform([home_team_name])[0]
    away_team_encoded = label_encoders['awayteam'].transform([away_team_name])[0]
    round_encoded = label_encoders['round'].transform([round_name])[0]
    venue_encoded = label_encoders['venue'].transform([venue_name])[0]

    input_full = pd.DataFrame({
        'year': [year],
        'round': [round_encoded],
        'hometeam': [home_team_encoded],
        'awayteam': [away_team_encoded],
        'venue': [venue_encoded],
        'maxtemp': [maxtemp],
        'mintemp': [mintemp],
        **{k: [v] for k, v in predicted_stats.items()}
    })

    home_score = model_home.predict(input_full)[0]
    away_score = model_away.predict(input_full)[0]

    print(f"Predicted Score:\n{home_team_name}: {int(round(home_score))}\n{away_team_name}: {int(round(away_score))}")
    return int(round(home_score)), int(round(away_score))

# ✅ Test
predict_match('Richmond', 'Carlton', 'R1', 'M.C.G.')


      gameid  year round        date       hometeam         awayteam  \
0  2018R0101  2018    R1  2018-03-22       Richmond          Carlton   
1  2018R0102  2018    R1  2018-03-23       Essendon         Adelaide   
2  2018R0103  2018    R1  2018-03-24       St Kilda   Brisbane Lions   
3  2018R0104  2018    R1  2018-03-24  Port Adelaide        Fremantle   
4  2018R0105  2018    R1  2018-03-24     Gold Coast  North Melbourne   

   hometeamscoreft  awayteamscoreft             venue  attendance  ...  \
0            17.19            15.50            M.C.G.     90151.0  ...   
1            14.15            12.15         Docklands     43016.0  ...   
2            16.11            12.10         Docklands     23731.0  ...   
3            16.14             9.60     Adelaide Oval     38324.0  ...   
4             7.13             5.90  Cazaly's Stadium      3722.0  ...   

   Kicks_away  Marks_away  Handballs_away Goals_away Behinds_away  \
0         207          88             169         15 

(89, 80)

In [ ]:
predict_match("Richmond", "Port Adelaide", "R1", "Adelaide Oval", year=2018, maxtemp=28.7, mintemp=14)

Predicted Score:
Richmond: 94
Port Adelaide: 73


(94, 73)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Step 1: Prepare test data for evaluation
# Important: test data must include both metadata and player stats
X_test_full = pd.concat([X_test[metadata_features], df_matches.loc[X_test.index, player_stats]], axis=1)

# Step 2: Predict on test set
y_pred_home = model_home.predict(X_test_full)
y_pred_away = model_away.predict(X_test_full)

# Step 3: Round predictions (optional for realism)
y_pred_home_int = np.round(y_pred_home)
y_pred_away_int = np.round(y_pred_away)

# Step 4: Calculate evaluation metrics

# Home score metrics
mae_home = mean_absolute_error(y_test_home, y_pred_home_int)
rmse_home = np.sqrt(mean_squared_error(y_test_home, y_pred_home_int))
r2_home = r2_score(y_test_home, y_pred_home)

# Away score metrics
mae_away = mean_absolute_error(y_test_away, y_pred_away_int)
rmse_away = np.sqrt(mean_squared_error(y_test_away, y_pred_away_int))
r2_away = r2_score(y_test_away, y_pred_away)

# Step 5: Print results
print(f"🏠 Home Score - MAE: {mae_home:.2f}, RMSE: {rmse_home:.2f}, R²: {r2_home:.4f}")
print(f"🧳 Away Score - MAE: {mae_away:.2f}, RMSE: {rmse_away:.2f}, R²: {r2_away:.4f}")


🏠 Home Score - MAE: 2.76, RMSE: 3.62, R²: 0.9803
🧳 Away Score - MAE: 2.74, RMSE: 3.80, R²: 0.9786
